# Suppl Fig 6 – per-dataset clustered heatmaps of MCT fractions

Panels A–I: Flomics, Chalasani, Ibarra, Moufarrej, Reggiardo, Roskams-Hieter, Tao, Toden, Zhu.

The published panels were made on 2 July 2025, before the notebook was committed. To reproduce them, this notebook reads the metadata table and the dataset labels **as they were in commit `8598ef0` (2025-07-02, "new figures")** and uses the heatmap layout of that time (default fonts, legends sorted by size, colour bar "cell fraction prediction"). `collection_subcenter` did not exist yet in that table and is taken from the current one. Set `USE_JULY_2025_METADATA = False` to use the current tables (needed after the re-run).

In [ ]:
# {dataset}: Flomics_2, chalasani_merged, chen, ibarra, moufarrej, reggiardo, roskams, tao, toden, zhu, decru
FRACTIONS_FILE = "/mnt/efs/home/gasole/projects/FL-cfRNAmeta/script/2026_10_26_scripts_review/inputs_test/deconv_results{dataset}/dec_cfRNAmeta_fractions_results.csv"
SAMPLE_LIST_FILE = "/mnt/efs/home/gasole/projects/FL-cfRNAmeta/script/2026_10_26_scripts_review/inputs_test/deconv_results{dataset}/sample_list.txt"

OUTPUT_DIR = "../../output/deconvolution"

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path[:0] = [".", ".."]
import deconv_common

OUT = Path(OUTPUT_DIR)
deconv_common.set_fonts()

import matplotlib.pyplot as plt
plt.rcdefaults()

In [ ]:
USE_JULY_2025_METADATA = True

if USE_JULY_2025_METADATA:
    metadata = deconv_common.load_metadata(git_commit="8598ef0")
    labels = deconv_common.load_labels(git_commit="8598ef0")
    metadata["collection_subcenter"] = deconv_common.load_metadata()["collection_subcenter"]
else:
    metadata = deconv_common.load_metadata()
    labels = deconv_common.load_labels()

per_dataset, cell_types = deconv_common.load_deconvolution(metadata, FRACTIONS_FILE, SAMPLE_LIST_FILE)
mcts = deconv_common.select_mcts(per_dataset, cell_types)
print("major cell types:", mcts)

In [ ]:
import heatmap_plot

META = ["dataset_short_name", "dataset_batch", "collection_center", "plasma_tubes_short_name",
        "centrifugation_step_1", "centrifugation_step_2", "biomaterial", "nucleic_acid_type",
        "rna_extraction_kit_short_name", "dnase", "library_prep_kit_short_name",
        "libraryselection", "cdna_library_type", "read_length", "instrument", "tissue",
        "phenotype", "stage", "disease_stage", "sex"]
PANELS = {"A": "Flomics_2", "B": "chalasani_merged", "C": "ibarra", "D": "moufarrej",
          "E": "reggiardo", "F": "roskams", "G": "tao", "H": "toden", "I": "zhu"}

for panel, key in PANELS.items():
    df = per_dataset[key].copy()
    df["dataset_batch"] = df["dataset_batch"].replace(labels)
    heatmap_plot.create_heatmap(
        df, mcts, META, metadata_colors_mapping=deconv_common.HEATMAP_COLORS_20250702,
        remove_all_zero=True, rescale_values=True, row_cluster=True,
        output=str(OUT / f"suppfig6{panel}_{key}"),
        row_height=8, legend_gap=5,
        heatmap_value_name="cell fraction prediction", legend_style="default")
    plt.close("all")